In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# المعاملات الشائعة الاستخدام في التحويل

<details>
<summary><b>إصدارات الحزم</b></summary>

تم تطوير الكود الموجود في هذه الصفحة باستخدام المتطلبات التالية.
ننصح باستخدام هذه الإصدارات أو ما هو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
تصف هذه الصفحة بعض المعاملات الأكثر استخدامًا في التحويل المحلي. يتم ضبط هذه المعاملات عبر الوسائط المُمرَّرة إلى [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) أو [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## درجة التقريب
يمكنك استخدام درجة التقريب لتحديد مدى قرب الدائرة الناتجة من الدائرة المطلوبة (المدخلة). وهي قيمة عشرية في النطاق (0.0 - 1.0)، حيث تعني 0.0 أقصى درجة تقريب وتعني 1.0 (الافتراضية) عدم التقريب تمامًا. تُقلل القيم الأصغر من دقة الإخراج في مقابل سهولة التنفيذ (أي عدد بوابات أقل). القيمة الافتراضية هي 1.0.

في تركيب الوحدوية ثنائية الكيوبت (المستخدمة في المراحل الأولى لجميع المستويات وفي مرحلة التحسين مع مستوى التحسين 3)، تحدد هذه القيمة الدقة المستهدفة لتحلل الإخراج. أي مقدار الخطأ الذي يُدخَل عند تحويل التمثيل المصفوفي للدائرة إلى بوابات منفصلة. إذا كانت درجة التقريب أقل (تقريب أكبر)، فستختلف الدائرة الناتجة من التركيب أكثر عن المصفوفة المدخلة، لكنها ستحتوي على الأرجح على عدد أقل من البوابات (لأن أي عملية ثنائية الكيوبت يمكن تحليلها بصورة مثالية بحد أقصى ثلاث بوابات CX)، مما يجعل تنفيذها أسهل.

عندما تكون درجة التقريب أقل من 1.0، قد يتم تركيب دوائر تحتوي على بوابة CX واحدة أو اثنتين، مما يُقلل الخطأ الناتج من العتاد لكنه يزيد الخطأ الناتج من التقريب. ونظرًا لأن CX هي أغلى بوابة من حيث الخطأ، فقد يكون من المفيد تقليل عددها على حساب الدقة في التركيب (استُخدمت هذه التقنية لزيادة الحجم الكمي على أجهزة IBM&reg;: [التحقق من صحة الحواسيب الكمومية باستخدام دوائر نماذج عشوائية](https://arxiv.org/abs/1811.12926)).

كمثال على ذلك، نُولّد بوابة `UnitaryGate` ثنائية الكيوبت عشوائية سيتم تركيبها في المرحلة الأولية. قد يُولّد ضبط `approximation_degree` بقيمة أقل من 1.0 دائرة تقريبية. يجب أيضًا تحديد `basis_gates` لإعلام أسلوب التركيب بالبوابات التي يمكنه استخدامها في التركيب التقريبي.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

ينتج عن ذلك `2` لأن التقريب يتطلب عددًا أقل من بوابات CX.

<span id="seed"></span>
## بذرة مولّد الأعداد العشوائية
بعض أجزاء المحوّل عشوائية الطابع، لذا قد تُعيد عمليات التحويل المتكررة نتائج مختلفة. للحصول على نتيجة قابلة للتكرار، يمكنك ضبط البذرة لمولد الأعداد الشبه‑عشوائية باستخدام الوسيطة `seed_transpiler`. ستُعيد عمليات التشغيل المتكررة التي تستخدم نفس البذرة النتائج ذاتها.

مثال:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## التخطيط الأولي
قبل التحويل، تكون الكيوبتات الموجودة في دائرتك كيوبتات افتراضية لا تتوافق بالضرورة مع الكيوبتات الفيزيائية على الـ Backend المستهدف. يمكنك تحديد التعيين الأولي للكيوبتات الافتراضية إلى الكيوبتات الفيزيائية باستخدام الوسيطة `initial_layout`. لاحظ أن تخطيط الكيوبت النهائي قد يختلف عن التخطيط الأولي لأن المحوّل قد يُعيد ترتيب الكيوبتات باستخدام بوابات التبديل أو وسائل أخرى.

في المثال أدناه، نُنشئ تخطيطًا أوليًا لـ Backend المحاكي [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) من خلال إنشاء كائن [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). يُعيّن تخطيطنا الكيوبت الأول من دائرتنا إلى الكيوبت 5 في Sherbrooke، ويُعيّن الكيوبت الثاني إلى الكيوبت 6 في Sherbrooke. لاحظ أن الكيوبتات الفيزيائية تُمثَّل دائمًا بأعداد صحيحة.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

بالإضافة إلى تحديد كائن Layout، يمكنك أيضًا تمرير قائمة من الأعداد الصحيحة، حيث يحتوي العنصر رقم $i$ من القائمة على الكيوبت الفيزيائي الذي ينبغي تعيين الكيوبت رقم $i$ إليه. مثلًا:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

يمكنك استخدام دالة [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) لإنشاء رسم تخطيطي لرسم الجهاز مع معلومات الخطأ وتسميات الكيوبتات الفيزيائية. يمكنك أيضًا الاطلاع على رسومات مماثلة في صفحة [موارد الحوسبة](https://quantum.cloud.ibm.com/computers).